In [6]:
import numpy as np

# ==============================================================
# 函数名：truss3d_element_stiffness
# 功能：计算三维杆单元长度、方向余弦、6×6全局刚度矩阵
# 输入：
#   x1, x2 : 节点1、节点2坐标 [x, y, z]
#   E      : 弹性模量（Pa）
#   A      : 横截面积（m²）
# 输出：
#   L       : 单元长度（m）
#   c       : 方向余弦 [cx, cy, cz]
#   Ke      : 6×6 全局刚度矩阵
# ==============================================================
def truss3d_element_stiffness(x1, x2, E, A):
    # 转为浮点数组，避免整数运算
    x1 = np.array(x1, dtype=float)
    x2 = np.array(x2, dtype=float)

    # 坐标差
    dx = x2[0] - x1[0]
    dy = x2[1] - x1[1]
    dz = x2[2] - x1[2]

    # 单元长度
    L = np.sqrt(dx**2 + dy**2 + dz**2)

    # 防退化：节点重合时报错
    if L < 1e-12:
        raise ValueError("错误：两节点重合，单元退化，无法计算！")

    # 方向余弦
    cx = dx / L
    cy = dy / L
    cz = dz / L

    # 刚度系数 EA/L
    EA_L = E * A / L

    # 6×6 三维杆单元刚度矩阵
    Ke = np.array([
        [ cx**2,  cx*cy, cx*cz, -cx**2, -cx*cy, -cx*cz],
        [ cx*cy,  cy**2, cy*cz, -cx*cy, -cy**2, -cy*cz],
        [ cx*cz,  cy*cz, cz**2, -cx*cz, -cy*cz, -cz**2],
        [-cx**2, -cx*cy, -cx*cz,  cx**2,  cx*cy,  cx*cz],
        [-cx*cy, -cy**2, -cy*cz,  cx*cy,  cy**2,  cy*cz],
        [-cx*cz, -cy*cz, -cz**2,  cx*cz,  cy*cz,  cz**2]
    ]) * EA_L

    return L, np.array([cx, cy, cz]), Ke

# ==============================================================
# 函数名：truss3d_element_stress
# 功能：由节点位移计算单元应变、应力、轴力
# 输入：
#   x1, x2, E, A : 单元几何与材料参数
#   de            : 节点位移向量 [u1, v1, w1, u2, v2, w2]
# 输出：
#   epsilon : 轴向应变
#   sigma   : 轴向应力（Pa）
#   N       : 轴力（N）
# ==============================================================
def truss3d_element_stress(x1, x2, E, A, de):
    L, c, _ = truss3d_element_stiffness(x1, x2, E, A)
    cx, cy, cz = c

    # 应变-位移矩阵（行向量）
    B = np.array([-cx, -cy, -cz, cx, cy, cz]) / L

    # 应变、应力、轴力
    epsilon = B @ de
    sigma   = E * epsilon
    N       = sigma * A

    return epsilon, sigma, N

# ==============================================================
# 函数名：check_matrix_properties
# 功能：检查刚度矩阵的对称性、奇异性、半正定性
# 输入：Ke：6×6刚度矩阵
# 输出：sym, singular, semidef, det, eigvals
# ==============================================================
def check_matrix_properties(Ke):
    # 对称性
    sym = np.allclose(Ke, Ke.T, atol=1e-10)
    # 行列式
    det = np.linalg.det(Ke)
    # 特征值
    eigvals = np.linalg.eigvalsh(Ke)
    # 奇异性
    singular = abs(det) < 1e-10
    # 半正定性
    semidef = np.all(eigvals >= -1e-10)
    #修正最小特征值
    return sym, singular, semidef, det, eigvals

# ==============================================================
# 函数名：test_case1
# 功能：算例1：沿x轴一维杆单元
# ==============================================================
def test_case1():
    print("=" * 60)
    print("                  算例1：沿x轴一维杆单元")
    print("=" * 60)

    # 单元参数
    x1 = [0, 0, 0]
    x2 = [2, 0, 0]
    E  = 200e9
    A  = 1.0e-4
    de = [0, 0, 0, 1.0e-3, 0, 0]

    # 计算刚度
    L, c, Ke = truss3d_element_stiffness(x1, x2, E, A)
    eps, sig, N = truss3d_element_stress(x1, x2, E, A, de)

    # 输出几何
    print(f"单元长度 L = {L:.2f} m")
    print(f"方向余弦 c = [{c[0]:.4f}, {c[1]:.4f}, {c[2]:.4f}]")
    print()

    # 输出刚度矩阵（仅非零项）
    print("刚度矩阵（仅非零项）：")
    Ke_2x2 = np.array([
        [Ke[0,0], Ke[0,3]],
        [Ke[3,0], Ke[3,3]]
    ])
    print(Ke_2x2)
    print()

    # 输出力学结果
    print(f"轴向应变    ε = {eps:.4e}")
    print(f"轴向应力    σ = {sig/1e6:.2f} MPa")
    print(f"轴力        N = {N:.2e} N")
    print()

    # 矩阵性质（每项单独一行）
    sym, sing, semid, det, eigv = check_matrix_properties(Ke)
    print("刚度矩阵性质：")
    print(f"  对称性    : {sym}")
    print(f"  奇异性    : {sing}")
    print(f"  半正定性  : {semid}")
    print(f"  行列式    : {det:.2e}")
    print(f"  最小特征值 : {np.min(eigv):.2e}")
    print()

# ==============================================================
# 函数名：test_case2
# 功能：算例2：空间任意方向杆单元
# ==============================================================
def test_case2():
    print("=" * 60)
    print("                算例2：空间任意方向杆单元")
    print("=" * 60)

    # 单元参数
    x1 = [0, 0, 0]
    x2 = [1, 2, 2]
    E  = 210e9
    A  = 2.0e-4
    de = [0, 0, 0, 1.0e-3, 2.0e-3, 2.0e-3]

    # 计算刚度
    L, c, Ke = truss3d_element_stiffness(x1, x2, E, A)
    eps, sig, N = truss3d_element_stress(x1, x2, E, A, de)

    # 输出几何
    print(f"单元长度 L = {L:.2f} m")
    print(f"方向余弦 c = [{c[0]:.4f}, {c[1]:.4f}, {c[2]:.4f}]")
    print()

    # 输出刚度矩阵（完整6×6）
    print("刚度矩阵（6×6）：")
    print(Ke)
    print()

    # 输出力学结果
    print(f"轴向应变    ε = {eps:.4e}")
    print(f"轴向应力    σ = {sig/1e6:.2f} MPa")
    print(f"轴力        N = {N:.2e} N")
    print()

    # 矩阵性质（每项单独一行）
    sym, sing, semid, det, eigv = check_matrix_properties(Ke)
    print("刚度矩阵性质：")
    print(f"  对称性    : {sym}")
    print(f"  奇异性    : {sing}")
    print(f"  半正定性  : {semid}")
    print(f"  行列式    : {det:.2e}")
    print(f"  最小特征值 : {np.min(eigv):.2e}")
    print()

    # 刚体位移验证
    print("刚体位移验证：")
    de_rigid = [0.1, 0.2, 0.3, 0.1, 0.2, 0.3]
    eps_r, sig_r, N_r = truss3d_element_stress(x1, x2, E, A, de_rigid)
    print(f"  应变 : {eps_r:.2e}")
    print(f"  应力 : {sig_r:.2e} Pa")
    print(f"  轴力 : {N_r:.2e} N")
    print()

# ==============================================================
# 函数名：verify_stiffness_meaning
# 功能：验证刚度矩阵物理意义：单位位移→力向量=矩阵对应列
# ==============================================================
def verify_stiffness_meaning():
    print("=" * 60)
    print("                刚度矩阵物理意义验证")
    print("=" * 60)

    x1 = [0,0,0]
    x2 = [1,2,2]
    E  = 210e9
    A  = 2.0e-4
    _, _, Ke = truss3d_element_stiffness(x1, x2, E, A)

    for j in [1, 2, 3]:
        de = np.zeros(6)
        de[j] = 1.0
        Fe = Ke @ de
        col = Ke[:, j]

        print(f"自由度 j={j} 单位位移")
        print(f"力向量 Fe : {Fe}")
        print(f"刚度矩阵第{j}列 : {col}")
        print(f"是否相等 : {np.allclose(Fe, col)}")
        print()

# ==============================================================
# 主程序入口
# ==============================================================
if __name__ == "__main__":
    test_case1()
    test_case2()
    verify_stiffness_meaning()

                  算例1：沿x轴一维杆单元
单元长度 L = 2.00 m
方向余弦 c = [1.0000, 0.0000, 0.0000]

刚度矩阵（仅非零项）：
[[ 10000000. -10000000.]
 [-10000000.  10000000.]]

轴向应变    ε = 5.0000e-04
轴向应力    σ = 100.00 MPa
轴力        N = 1.00e+04 N

刚度矩阵性质：
  对称性    : True
  奇异性    : True
  半正定性  : True
  行列式    : 0.00e+00
  最小特征值 : 0.00e+00

                算例2：空间任意方向杆单元
单元长度 L = 3.00 m
方向余弦 c = [0.3333, 0.6667, 0.6667]

刚度矩阵（6×6）：
[[ 1555555.55555556  3111111.11111111  3111111.11111111 -1555555.55555556
  -3111111.11111111 -3111111.11111111]
 [ 3111111.11111111  6222222.22222222  6222222.22222222 -3111111.11111111
  -6222222.22222222 -6222222.22222222]
 [ 3111111.11111111  6222222.22222222  6222222.22222222 -3111111.11111111
  -6222222.22222222 -6222222.22222222]
 [-1555555.55555556 -3111111.11111111 -3111111.11111111  1555555.55555556
   3111111.11111111  3111111.11111111]
 [-3111111.11111111 -6222222.22222222 -6222222.22222222  3111111.11111111
   6222222.22222222  6222222.22222222]
 [-3111111.11111111 -6222222.2